# DF40 Dataset — Extraction Optimisée

# CELLULE 1 — MONTAGE DU DRIVE ET CONFIGURATION

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

!pip install tqdm pillow matplotlib pandas numpy -q

import os
import zipfile
import shutil
import random
from pathlib import Path
from collections import defaultdict
from tqdm import tqdm
import matplotlib.pyplot as plt
from PIL import Image
import numpy as np
import pandas as pd
from datetime import datetime

# Seed de reproductibilité
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

# Chemins du projet
PROJECT_ROOT  = '/content/drive/MyDrive/Memoire_Deepfakes'
DATA_DIR      = f'{PROJECT_ROOT}/data'
RAW_DIR       = f'{DATA_DIR}/raw'
FAKE_TEMP_DIR = f'{RAW_DIR}/DF40_temp/fake'
REAL_TEMP_DIR = f'{RAW_DIR}/DF40_temp/real'
FAKE_OUT_DIR  = f'{RAW_DIR}/DF40_fake_DMs'
REAL_OUT_DIR  = f'{RAW_DIR}/DF40_real'

# Paramètres FAKE
DM_CONFIG = {
    'MidJourney' : 1_600,
    'ddim'       : 1_300,
    'DiT'        :   358,
    'SiT'        :   258,
    'CollabDiff' :   258, #sera enlevé plus tard suite à l'audit
}
assert sum(DM_CONFIG.values()) == 3_774, "FAKE total ≠ 3 774"

# Paramètres REAL
FRAMES_PER_VIDEO = 2
SOURCE_CONFIGS = {
    'FaceForensics++' : {'prefix': 'FF',      'offset': 0},
    'Celeb-DF-v2'     : {'prefix': 'CelebDF', 'offset': 10_000},
}

TOTAL_FAKE  = sum(DM_CONFIG.values())
REAL_ESTIMATED = 999 * FRAMES_PER_VIDEO + 888 * FRAMES_PER_VIDEO   # ~3 774
GRAND_TOTAL = REAL_ESTIMATED + TOTAL_FAKE

# output
print("NOTEBOOK 01 — EXTRACTION AVEC SPLIT VIDÉO-LEVEL")
print(f"\n  Architecture :")
print(f"    Frames/vidéo  : {FRAMES_PER_VIDEO} (espacement uniforme)")
print(f"    Nommage REAL  : {{prefix}}_vid{{id:04d}}_f{{frame:02d}}.jpg")
print(f"    Nommage FAKE  : {{method}}_{{counter:06d}}.jpg")
print(f"\n  Distribution FAKE :")
for method, n in DM_CONFIG.items():
    bar = '█' * (n // 80)
    print(f"    {method:<15s} : {n:>5,}  {bar}")
print(f"\n  TOTAL FAKE estimé  : {TOTAL_FAKE:,}")
print(f"  TOTAL REAL estimé  : {REAL_ESTIMATED:,}")
print(f"  GRAND TOTAL estimé : {GRAND_TOTAL:,}")
print(f"  Ratio F/R          : {TOTAL_FAKE / REAL_ESTIMATED:.4f}")
print(f"\n  Méthodes exclues   : sd2.1, pixart")
print("\n Configuration chargée")

# CELLULE 2 — VÉRIFICATION DES FICHIERS SOURCE

In [ ]:
print("Vérification des fichiers .zip source...\n")

fake_found = []
real_found = []

print("FAKE (méthodes retenues) :")
for method_name in DM_CONFIG.keys():
    fname = f"{method_name}.zip"
    path  = f"{FAKE_TEMP_DIR}/{fname}"
    if os.path.exists(path):
        size_gb = os.path.getsize(path) / 1e9
        print(f"  {fname:<25s} ({size_gb:.2f} GB)")
        fake_found.append(fname)
    else:
        print(f"  {fname:<25s} MANQUANT")

print("\n FAKE (méthodes exclues — informatif) :")
for fname in ['sd2.1.zip', 'pixart.zip']:
    path   = f"{FAKE_TEMP_DIR}/{fname}"
    status = "présent (ignoré)" if os.path.exists(path) else "absent"
    print(f"  ⊘  {fname:<25s} {status}")

print("\n REAL :")
expected_real = [
    'FaceForensics++_real_data_for_DF40.zip',
    'Celeb-DF-v2_real_data_for_DF40.zip',
]
for fname in expected_real:
    path = f"{REAL_TEMP_DIR}/{fname}"
    if os.path.exists(path):
        size_gb = os.path.getsize(path) / 1e9
        print(f"  {fname:<55s} ({size_gb:.2f} GB)")
        real_found.append(fname)
    else:
        print(f"  {fname:<55s} MANQUANT")

print(f"\n  {len(fake_found)}/{len(DM_CONFIG)} FAKE + {len(real_found)}/2 REAL prêts")

# CELLULE 3 — EXTRACTION FAKE

In [ ]:
def extract_sample_from_zip(zip_path, output_dir, max_images):
    """
    Extrait un échantillon aléatoire d'images depuis un .zip.
    Nommage par compteur global pour garantir l'unicité des noms.
    """
    method_name = os.path.basename(zip_path).replace('.zip', '')

    # Lister les images dans le .zip
    with zipfile.ZipFile(zip_path, 'r') as zf:
        all_files   = zf.namelist()
        image_files = [
            f for f in all_files
            if f.lower().endswith(('.png', '.jpg', '.jpeg'))
            and not f.startswith('__MACOSX')
        ]

    n_available  = len(image_files)
    n_to_extract = min(max_images, n_available)
    print(f"  Disponibles : {n_available:,}  →  Sélectionnés : {n_to_extract:,}")

    if n_to_extract == 0:
        return 0

    selected = random.sample(image_files, n_to_extract)

    # Extraction vers SSD local
    local_temp = f"/content/temp_{method_name}"
    os.makedirs(local_temp, exist_ok=True)

    with zipfile.ZipFile(zip_path, 'r') as zf:
        for entry in tqdm(selected, desc=f"  Extraction {method_name[:12]}",
                          unit="img", leave=False):
            try:
                zf.extract(entry, local_temp)
            except Exception:
                pass

    # Copie vers Drive — nommage {method}_{counter:06d}.jpg
    final_dir = os.path.join(output_dir, method_name)
    os.makedirs(final_dir, exist_ok=True)

    counter = 0
    for root, _, files in os.walk(local_temp):
        for f in files:
            if f.lower().endswith(('.png', '.jpg', '.jpeg')):
                src = os.path.join(root, f)
                dst = os.path.join(final_dir,
                                   f"{method_name}_{counter:06d}.jpg")
                shutil.copy2(src, dst)
                counter += 1

    shutil.rmtree(local_temp)
    return counter

print("Fonction d'extraction FAKE définie")

os.makedirs(FAKE_OUT_DIR, exist_ok=True)

print("EXTRACTION FAKE")

fake_stats = {}

for method_name, n_images in DM_CONFIG.items():
    zip_path = f"{FAKE_TEMP_DIR}/{method_name}.zip"

    if not os.path.exists(zip_path):
        print(f"\n{method_name} : zip introuvable")
        fake_stats[method_name] = 0
        continue

    print(f"\n{method_name}  (cible : {n_images:,})")
    count = extract_sample_from_zip(zip_path, FAKE_OUT_DIR, n_images)
    fake_stats[method_name] = count
    print(f"  {count:,} images extraites")

print("\n" + "=" * 70)
print("RÉSUMÉ FAKE")
print("=" * 70)
for method, cible in DM_CONFIG.items():
    cnt    = fake_stats.get(method, 0)
    status = 'OK' if cnt >= cible * 0.95 else 'Warning'
    print(f"  {status} {method:<15s} : {cnt:>5,}  (cible : {cible:,})")

total_fake_extracted = sum(fake_stats.values())
print(f"  {'TOTAL FAKE':15s} : {total_fake_extracted:>5,}  (cible : {TOTAL_FAKE:,})")


# CELLULE 4 — EXTRACTION REAL (split vidéo-level)

In [ ]:
def extract_real_video_level(zip_path, output_dir, source_name,
                              prefix, frames_per_video=2):
    """
    Extrait FRAMES_PER_VIDEO frames par vidéo depuis un .zip REAL.
    Encode le video_id dans le nom de fichier pour le split vidéo-level.

    Format de sortie : {prefix}_vid{video_id:04d}_f{frame_idx:02d}.jpg
    """
    print(f"\n{source_name}")

    # Lister et grouper par dossier parent (= une vidéo)
    with zipfile.ZipFile(zip_path, 'r') as zf:
        all_files   = zf.namelist()
        image_files = [
            f for f in all_files
            if f.lower().endswith(('.png', '.jpg', '.jpeg'))
            and not f.startswith('__MACOSX')
        ]

    print(f"  Frames totales disponibles  : {len(image_files):,}")

    groups = defaultdict(list)
    for entry in image_files:
        groups[os.path.dirname(entry)].append(entry)

    n_videos = len(groups)
    print(f"  Vidéos/identités uniques    : {n_videos:,}")
    print(f"  Frames extraites par vidéo  : {frames_per_video}")
    print(f"  Total images attendu        : {n_videos * frames_per_video:,}")

    # Sélection des entrées à extraire
    # Pour chaque vidéo : espacement uniforme sur les frames triées
    entries_to_extract = []   # liste de (entry_zip_path, video_id, frame_idx)

    for video_id, (group_key, video_entries) in enumerate(groups.items()):
        video_sorted = sorted(video_entries)
        n = len(video_sorted)
        k = min(frames_per_video, n)

        indices = [int(i * (n - 1) / (k - 1)) for i in range(k)] if k > 1 else [0]

        for frame_idx, entry_idx in enumerate(indices):
            entries_to_extract.append((
                video_sorted[entry_idx],   # chemin complet dans le zip
                video_id,
                frame_idx
            ))

    # Extraction vers SSD local
    local_temp = f"/content/temp_real_{source_name}"
    os.makedirs(local_temp, exist_ok=True)

    entries_only = [e[0] for e in entries_to_extract]
    with zipfile.ZipFile(zip_path, 'r') as zf:
        for entry in tqdm(entries_only,
                          desc=f"  Extraction {source_name[:15]}",
                          unit="img", leave=False):
            try:
                zf.extract(entry, local_temp)
            except Exception:
                pass

    # Copie vers Drive avec nommage vidéo-level
    #
    # FIX v4 : src_path construit directement depuis l'entrée zip.
    # zf.extract(entry, local_temp) crée exactement local_temp/entry.
    # On n'a donc pas besoin de chercher le fichier — son chemin est connu.

    video_manifest = []
    images_copied  = 0
    current_video_id = -1
    current_frames   = []

    for entry, video_id, frame_idx in tqdm(entries_to_extract,
                                            desc=f"  Copie {source_name[:15]}",
                                            unit="img", leave=False):

        # Chemin source exact — reconstruit depuis l'entrée zip
        src_path = os.path.join(local_temp, entry)

        if not os.path.exists(src_path):
            continue

        dst_name = f"{prefix}_vid{video_id:04d}_f{frame_idx:02d}.jpg"
        dst_path = os.path.join(output_dir, dst_name)

        shutil.copy2(src_path, dst_path)
        images_copied += 1

        # Construction du manifeste
        if video_id != current_video_id:
            if current_video_id >= 0:
                video_manifest.append({
                    'video_id'       : current_video_id,
                    'source'         : source_name,
                    'prefix'         : prefix,
                    'original_folder': os.path.dirname(entry),
                    'n_frames'       : len(current_frames),
                    'files'          : current_frames,
                })
            current_video_id = video_id
            current_frames   = []

        current_frames.append(dst_name)

    # Dernier groupe
    if current_video_id >= 0 and current_frames:
        video_manifest.append({
            'video_id'       : current_video_id,
            'source'         : source_name,
            'prefix'         : prefix,
            'original_folder': '',
            'n_frames'       : len(current_frames),
            'files'          : current_frames,
        })

    shutil.rmtree(local_temp)

    n_videos_done = len(video_manifest)
    print(f"  {images_copied:,} images copiées depuis {n_videos_done:,} vidéos")

    return images_copied, video_manifest


print("Fonction d'extraction REAL v4 définie (fix copie)")
print()
print("  Fix : src_path = os.path.join(local_temp, entry)")
print("  Format  : {prefix}_vid{id:04d}_f{frame:02d}.jpg")
print("  Ex FF++ : FF_vid0000_f00.jpg ↔ FF_vid0000_f01.jpg = même vidéo")

# CELLULE 5 — LANCEMENT EXTRACTION REAL et SAUVEGARDE DU MANIFESTE

In [ ]:
# Le manifeste vidéo est un CSV listant chaque vidéo, sa source, et les noms de fichiers des frames extraites.
# Il est sauvegardé dans data/ et sera lu par le notebook 03 (split) pour réaliser le split au niveau vidéo sans ré-ouvrir les zips.

os.makedirs(REAL_OUT_DIR, exist_ok=True)

print("=" * 70)
print("EXTRACTION REAL (split vidéo-level v4)")
print("=" * 70)

all_manifests    = []
total_real_copied = 0

for fname in real_found:
    zip_path    = f"{REAL_TEMP_DIR}/{fname}"
    source_name = fname.replace('_real_data_for_DF40.zip', '')
    cfg         = SOURCE_CONFIGS.get(
        source_name,
        {'prefix': source_name[:8], 'offset': 10_000}
    )

    n_copied, manifest = extract_real_video_level(
      zip_path        = zip_path,
      output_dir      = REAL_OUT_DIR,
      source_name     = source_name,
      prefix          = cfg['prefix'],
      frames_per_video= FRAMES_PER_VIDEO
    )


# Sauvegarde du manifeste vidéo
manifest_records = []
for entry in all_manifests:
    for fname_img in entry['files']:
        manifest_records.append({
            'filename'        : fname_img,
            'video_id'        : entry['video_id'],
            'source'          : entry['source'],
            'prefix'          : entry['prefix'],
            'original_folder' : entry['original_folder'],
            'label'           : 0,   # 0 = REAL
        })

df_manifest = pd.DataFrame(manifest_records)
manifest_path = f"{DATA_DIR}/real_video_manifest.csv"
df_manifest.to_csv(manifest_path, index=False)


print("RÉSUMÉ EXTRACTION REAL")
print(f"  Total images copiées : {total_real_copied:,}")
print()

for source_name, cfg in SOURCE_CONFIGS.items():
    prefix = cfg['prefix']
    files  = [f for f in os.listdir(REAL_OUT_DIR)
               if f.startswith(f"{prefix}_vid")]
    vids   = len(set(f.split('_f')[0] for f in files))
    print(f"  {source_name:<20s} : {len(files):>5,} images  ({vids:,} vidéos)")

print()
print(f"  Manifeste vidéo sauvegardé : {manifest_path}")
print(f"  Lignes dans le manifeste   : {len(df_manifest):,}")
print()
print("  ⚠️  Ce fichier est requis par le notebook 03 (split).")
print("     Ne pas le supprimer.")
print("=" * 70)

EXTRACTION REAL (split vidéo-level v4)

📂 FaceForensics++
  Frames totales disponibles  : 31,949
  Vidéos/identités uniques    : 999
  Frames extraites par vidéo  : 2
  Total images attendu        : 1,998


  ✅ 1,998 images copiées depuis 999 vidéos

📂 Celeb-DF-v2
  Frames totales disponibles  : 28,174
  Vidéos/identités uniques    : 888
  Frames extraites par vidéo  : 2
  Total images attendu        : 1,776


  ✅ 1,776 images copiées depuis 888 vidéos

RÉSUMÉ EXTRACTION REAL
  Total images copiées : 0

  FaceForensics++      : 1,998 images  (999 vidéos)
  Celeb-DF-v2          : 1,776 images  (888 vidéos)

  Manifeste vidéo sauvegardé : /content/drive/MyDrive/Memoire_Deepfakes/data/real_video_manifest.csv
  Lignes dans le manifeste   : 0

  ⚠️  Ce fichier est requis par le notebook 03 (split).
     Ne pas le supprimer.


# CELLULE 5-BIS — RECONSTRUCTION DU MANIFESTE DEPUIS LES FICHIERS COPIÉS

In [ ]:
import re
import pandas as pd

print("Reconstruction du manifeste vidéo depuis les fichiers copiés...")

# Regex pour parser le nommage vidéo-level
PATTERN = re.compile(r'^(FF|CelebDF)_vid(\d{4})_f(\d{2})\.jpg$')

SOURCE_MAP = {
    'FF'      : 'FaceForensics++',
    'CelebDF' : 'Celeb-DF-v2',
}

manifest_records = []

all_real_files = sorted(os.listdir(REAL_OUT_DIR))
print(f"  Fichiers trouvés dans REAL_OUT_DIR : {len(all_real_files):,}")

skipped = 0
for fname in all_real_files:
    m = PATTERN.match(fname)
    if not m:
        skipped += 1
        continue

    prefix    = m.group(1)
    video_id  = int(m.group(2))
    frame_idx = int(m.group(3))

    manifest_records.append({
        'filename'  : fname,
        'video_id'  : video_id,
        'source'    : SOURCE_MAP.get(prefix, prefix),
        'prefix'    : prefix,
        'frame_idx' : frame_idx,
        'label'     : 0,   # 0 = REAL
    })

if skipped > 0:
    print(f"  ⚠️  {skipped} fichiers ignorés (ne correspondent pas au pattern)")

df_manifest = pd.DataFrame(manifest_records)
df_manifest = df_manifest.sort_values(['prefix', 'video_id', 'frame_idx']).reset_index(drop=True)

# Vérifications
n_videos_ff    = df_manifest[df_manifest['prefix'] == 'FF']['video_id'].nunique()
n_videos_celeb = df_manifest[df_manifest['prefix'] == 'CelebDF']['video_id'].nunique()
n_frames_total = len(df_manifest)

print(f"\n  Lignes dans le manifeste : {n_frames_total:,}")
print(f"  Vidéos FF++              : {n_videos_ff:,}")
print(f"  Vidéos CelebDF-v2        : {n_videos_celeb:,}")
print(f"  Frames/vidéo moyenne     : {n_frames_total / (n_videos_ff + n_videos_celeb):.2f}")

if n_frames_total == 0:
    print("\n  ❌ MANIFESTE VIDE — Vérifier le contenu de REAL_OUT_DIR")
else:
    manifest_path = f"{DATA_DIR}/real_video_manifest.csv"
    df_manifest.to_csv(manifest_path, index=False)
    print(f"\n  ✅ Manifeste sauvegardé : {manifest_path}")
    print(f"\n  Aperçu (5 premières lignes) :")
    print(df_manifest.head().to_string(index=False))

# Vérification critique : chaque video_id doit avoir exactement FRAMES_PER_VIDEO frames
frames_per_vid = df_manifest.groupby(['prefix', 'video_id']).size()
anomalies = frames_per_vid[frames_per_vid != FRAMES_PER_VIDEO]
if len(anomalies) > 0:
    print(f"\n  ⚠️  {len(anomalies)} vidéos avec un nombre de frames anormal :")
    print(anomalies.head(10))
else:
    print(f"\n  ✅ Toutes les vidéos ont exactement {FRAMES_PER_VIDEO} frames — cohérence parfaite")

Reconstruction du manifeste vidéo depuis les fichiers copiés...
  Fichiers trouvés dans REAL_OUT_DIR : 3,774

  Lignes dans le manifeste : 3,774
  Vidéos FF++              : 999
  Vidéos CelebDF-v2        : 888
  Frames/vidéo moyenne     : 2.00

  ✅ Manifeste sauvegardé : /content/drive/MyDrive/Memoire_Deepfakes/data/real_video_manifest.csv

  Aperçu (5 premières lignes) :
               filename  video_id      source  prefix  frame_idx  label
CelebDF_vid0000_f00.jpg         0 Celeb-DF-v2 CelebDF          0      0
CelebDF_vid0000_f01.jpg         0 Celeb-DF-v2 CelebDF          1      0
CelebDF_vid0001_f00.jpg         1 Celeb-DF-v2 CelebDF          0      0
CelebDF_vid0001_f01.jpg         1 Celeb-DF-v2 CelebDF          1      0
CelebDF_vid0002_f00.jpg         2 Celeb-DF-v2 CelebDF          0      0

  ✅ Toutes les vidéos ont exactement 2 frames — cohérence parfaite


# CELLULE 6 — AUDIT FINAL

In [ ]:
def count_images(directory):
    exts = {'.png', '.jpg', '.jpeg'}
    return sum(
        1 for _, _, files in os.walk(directory)
        for f in files if Path(f).suffix.lower() in exts
    )

print("AUDIT FINAL — COMPTAGE RÉEL SUR DRIVE")

# FAKE
print("\n FAKE par méthode :")

fake_counts = {}
for method in DM_CONFIG.keys():
    method_path = os.path.join(FAKE_OUT_DIR, method)
    cnt = count_images(method_path) if os.path.isdir(method_path) else 0
    fake_counts[method] = cnt
    status = 'OK' if cnt >= DM_CONFIG[method] * 0.95 else 'Warning'
    print(f"  {status} {method:<15s} : {cnt:>5,}  (cible : {DM_CONFIG[method]:,})")
total_fake_audit = sum(fake_counts.values())

print(f"  {'TOTAL FAKE':15s} : {total_fake_audit:>5,}")

# REAL
print("\n REAL par source :")

real_counts = {}
for source_name, cfg in SOURCE_CONFIGS.items():
    prefix = cfg['prefix']
    files  = [f for f in os.listdir(REAL_OUT_DIR)
               if f.startswith(f"{prefix}_vid")]
    vids   = len(set(f.split('_f')[0] for f in files))
    real_counts[source_name] = len(files)
    print(f"  {source_name:<20s} : {len(files):>5,} images  ({vids:,} vidéos)")
total_real_audit = sum(real_counts.values())
print(f"  {'TOTAL REAL':15s} : {total_real_audit:>5,}")

# Résumé
grand_total = total_real_audit + total_fake_audit
ratio = total_fake_audit / total_real_audit if total_real_audit > 0 else 0

print("=============================================================")
print("RÉSUMÉ GLOBAL")
print(f"  Images REAL  : {total_real_audit:>8,}  (estimé : {REAL_ESTIMATED:,})")
print(f"  Images FAKE  : {total_fake_audit:>8,}  (cible  : {TOTAL_FAKE:,})")
print(f"  TOTAL        : {grand_total:>8,}  (estimé : {GRAND_TOTAL:,})")
print(f"  Ratio F/R    : {ratio:>8.4f}  (idéal  : 1.0000)")
print()
if 0.90 <= ratio <= 1.10:
    print("PARFAITEMENT ÉQUILIBRÉ")
    balance = "EXCELLENT"
elif 0.80 <= ratio <= 1.20:
    print("Acceptable")
    balance = "BON"
else:
    print("Déséquilibre à corriger")
    balance = "À AJUSTER"

# CELLULE 7 — VISUALISATION

In [ ]:
# Vérification visuelle du nommage vidéo-level :
# Les frames d'une même vidéo (même video_id) doivent représenter des moments DIFFÉRENTS (pose, éclairage) — confirme l'espacement.

def load_img(path, size=(128, 128)):
    try:
        return np.array(Image.open(path).convert('RGB').resize(size))
    except Exception:
        return np.zeros((size[1], size[0], 3), dtype=np.uint8)

# Vérification espacement intra-vidéo
print("Vérification de l'espacement intra-vidéo (3 vidéos aléatoires)\n")

all_real_files = sorted(os.listdir(REAL_OUT_DIR))
vid_ids = sorted(set(
    f.split('_f')[0] for f in all_real_files if '_vid' in f
))
sample_vids = random.sample(vid_ids, min(3, len(vid_ids)))

fig, axes = plt.subplots(len(sample_vids), FRAMES_PER_VIDEO,
                          figsize=(FRAMES_PER_VIDEO * 3, len(sample_vids) * 3))
if len(sample_vids) == 1:
    axes = [axes]
fig.suptitle(
    f'Vérification espacement intra-vidéo ({FRAMES_PER_VIDEO} frames/vidéo)\n'
    f'Chaque ligne = même vidéo — les frames doivent être visuellement différentes',
    fontsize=11, fontweight='bold'
)

for row_idx, vid_stem in enumerate(sample_vids):
    frames = sorted([
        f for f in all_real_files if f.startswith(vid_stem + '_f')
    ])
    for col_idx in range(FRAMES_PER_VIDEO):
        ax = axes[row_idx][col_idx] if FRAMES_PER_VIDEO > 1 else axes[row_idx]
        if col_idx < len(frames):
            img_path = os.path.join(REAL_OUT_DIR, frames[col_idx])
            ax.imshow(load_img(img_path))
            ax.set_title(frames[col_idx], fontsize=6)
        else:
            ax.imshow(np.zeros((128, 128, 3), dtype=np.uint8))
        ax.axis('off')
    if hasattr(axes[row_idx], '__len__'):
        axes[row_idx][0].set_ylabel(
            vid_stem.split('_vid')[0] + '\n' + vid_stem.split('_vid')[1],
            fontsize=8, rotation=0, labelpad=55, va='center'
        )

plt.tight_layout()
plt.show()

# Grille FAKE par méthode
fig, axes = plt.subplots(len(DM_CONFIG), 5,
                          figsize=(14, len(DM_CONFIG) * 2.5))
fig.suptitle('Échantillons FAKE par méthode DM', fontsize=13,
             fontweight='bold', y=1.01)

for row_idx, method in enumerate(DM_CONFIG.keys()):
    method_dir = os.path.join(FAKE_OUT_DIR, method)
    method_files = [
        os.path.join(method_dir, f)
        for f in os.listdir(method_dir)
        if f.lower().endswith(('.png', '.jpg', '.jpeg'))
    ] if os.path.isdir(method_dir) else []

    samples = random.sample(method_files, min(5, len(method_files)))
    for col_idx in range(5):
        ax = axes[row_idx, col_idx]
        if col_idx < len(samples):
            ax.imshow(load_img(samples[col_idx]))
        else:
            ax.imshow(np.zeros((128, 128, 3), dtype=np.uint8))
        ax.axis('off')
        if col_idx == 0:
            ax.set_ylabel(method, fontsize=9, rotation=0,
                          labelpad=65, va='center')

plt.tight_layout()
plt.show()
print("Visualisation terminée")

Output hidden; open in https://colab.research.google.com to view.

# CELLULE 8 — RAPPORT D'AUDIT ET ESPACE DISQUE

In [ ]:
# Rapport
report_path = (f"{DATA_DIR}/audit_report"
               f"{datetime.now().strftime('%Y%m%d_%H%M%S')}.txt")

with open(report_path, 'w', encoding='utf-8') as f:
    f.write("=" * 70 + "\n")
    f.write("RAPPORT D'AUDIT — DATASET DF40 v4\n")
    f.write("Mémoire : Obsolescence des détecteurs de deepfakes\n")
    f.write("Auteur  : Maxime Ducarme\n")
    f.write("=" * 70 + "\n")
    f.write(f"Date      : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
    f.write(f"Version   : v4 (split vidéo-level)\n")
    f.write(f"Seed      : {RANDOM_SEED}\n\n")

    f.write("1. ARCHITECTURE v4\n")
    f.write("-" * 70 + "\n")
    f.write(f"  Frames/vidéo REAL : {FRAMES_PER_VIDEO}\n")
    f.write(f"  Nommage REAL      : {{prefix}}_vid{{id:04d}}_f{{frame:02d}}.jpg\n")
    f.write(f"  Manifeste vidéo   : data/real_video_manifest.csv\n")
    f.write(f"  Data leakage      : ÉLIMINÉ (split vidéo-level dans nb03)\n\n")

    f.write("2. DONNÉES FAKE\n")
    f.write("-" * 70 + "\n")
    for method, cible in DM_CONFIG.items():
        cnt = fake_counts.get(method, 0)
        f.write(f"  {method:<15s} : {cnt:>5,}  (cible : {cible:,})\n")
    f.write(f"  Méthodes exclues  : sd2.1, pixart\n")
    f.write("-" * 70 + "\n")
    f.write(f"  TOTAL FAKE        : {total_fake_audit:>5,}\n\n")

    f.write("3. DONNÉES REAL\n")
    f.write("-" * 70 + "\n")
    for source_name, cfg in SOURCE_CONFIGS.items():
        prefix = cfg['prefix']
        files  = [f for f in os.listdir(REAL_OUT_DIR)
                   if f.startswith(f"{prefix}_vid")]
        vids   = len(set(f.split('_f')[0] for f in files))
        f.write(f"  {source_name:<20s} : {len(files):>5,} images "
                f"({vids:,} vidéos × {FRAMES_PER_VIDEO} frames)\n")
    f.write("-" * 70 + "\n")
    f.write(f"  TOTAL REAL        : {total_real_audit:>5,}\n\n")

    f.write("4. RÉSUMÉ\n")
    f.write("-" * 70 + "\n")
    f.write(f"  REAL  : {total_real_audit:>8,}\n")
    f.write(f"  FAKE  : {total_fake_audit:>8,}\n")
    f.write(f"  TOTAL : {grand_total:>8,}\n")
    f.write(f"  Ratio : {ratio:.4f}  —  {balance}\n\n")

    f.write("5. PROCHAINES ÉTAPES\n")
    f.write("-" * 70 + "\n")
    f.write("  [ ] Notebook 02 : Audit rigoureux (D1-D6)\n")
    f.write("  [ ] Notebook 03 : Split vidéo-level (lire real_video_manifest.csv)\n")
    f.write("  [ ] Notebook 04 : Inférence (4 détecteurs × 3 sets)\n")
    f.write("=" * 70 + "\n")

print(f"✅ Rapport : {report_path}\n")
with open(report_path) as f:
    print(f.read())

# Espace disque
print("=" * 70)
print("ESPACE DISQUE")
print("=" * 70)
for name, path in [('FAKE', FAKE_OUT_DIR), ('REAL', REAL_OUT_DIR)]:
    size = sum(
        os.path.getsize(os.path.join(r, f))
        for r, _, files in os.walk(path) for f in files
        if os.path.exists(os.path.join(r, f))
    )
    print(f"  {name:<6s} : {size / 1e9:.2f} GB")

print()
print("✅ Notebook 01 v4 terminé")
print("   Fichier clé généré : data/real_video_manifest.csv")
print("   Prochaine étape    : 02_data_audit.ipynb")

✅ Rapport : /content/drive/MyDrive/Memoire_Deepfakes/data/audit_report_v4_20260317_212142.txt

RAPPORT D'AUDIT — DATASET DF40 v4
Mémoire : Obsolescence des détecteurs de deepfakes
Auteur  : Maxime Ducarme
Date      : 2026-03-17 21:21:42
Version   : v4 (split vidéo-level)
Seed      : 42

1. ARCHITECTURE v4
----------------------------------------------------------------------
  Frames/vidéo REAL : 2
  Nommage REAL      : {prefix}_vid{id:04d}_f{frame:02d}.jpg
  Manifeste vidéo   : data/real_video_manifest.csv
  Data leakage      : ÉLIMINÉ (split vidéo-level dans nb03)

2. DONNÉES FAKE
----------------------------------------------------------------------
  MidJourney      : 1,600  (cible : 1,600)
  ddim            : 1,300  (cible : 1,300)
  DiT             :   358  (cible : 358)
  SiT             :   258  (cible : 258)
  CollabDiff      :   258  (cible : 258)
  Méthodes exclues  : sd2.1, pixart
----------------------------------------------------------------------
  TOTAL FAKE        : 3